# 3. Deep ensembles

Train `M` networks from different random starting points and treat the `M`
answers as your samples. There is no variational family, no KL term and nothing
to derive, and this is repeatedly the strongest baseline in the literature.

You write `fit_ensemble` and `predict_ensemble`.

About 1 hour.

## 3.1 What the samples mean here

Each member is a separate maximum-a-posteriori estimate that landed in a
different basin of the loss surface, and the mixture over members is a rough
approximation to the posterior predictive integral. It is not a posterior in any
strict sense: nothing weights the members, and their spread is whatever
initialisation and stochastic gradient descent happened to produce.

Why it works anyway is still argued about. The usual account is that different
initialisations find genuinely different functions, which a mean-field
approximation confined to one basin cannot represent. Keep that in mind when you
compare against notebook 04: if the principled method loses, that is a result
about the shape of the posterior, not a bug.

The cost is the obvious one. Training is `M` times more expensive, and unlike MC
Dropout you cannot buy more samples afterwards without training more networks.

In [ ]:
import sys
import time

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import torch

from bdl.data import load_track, make_toy1d, toy1d_grid
from bdl.metrics import evaluate, interval_coverage, results_table
from bdl.models import (
    build_model,
    count_parameters,
    default_loss,
    fit,
    gaussian_head,
    set_seed,
    track_hparams,
)
from bdl.plots import plot_band, plot_reliability, plot_uncertainty_vs_x
from bdl.store import load_run, save_run

%matplotlib inline

ds = make_toy1d()
grid = toy1d_grid()


# The functions you wrote in notebooks 01 and 02, repeated so this notebook stands
# on its own. The last two are only used on Track C.
def calibration_error(mu, sigma, y, n_bins=15):
    levels = torch.linspace(0.0, 1.0, n_bins + 2)[1:-1]
    gaps = [abs(interval_coverage(mu, sigma, y, float(q)) - float(q)) for q in levels]
    return float(np.mean(gaps))


def decompose_variance(mu, sigma):
    aleatoric = (sigma**2).mean(dim=0)
    epistemic = mu.var(dim=0, unbiased=False)
    return aleatoric + epistemic, aleatoric, epistemic


def decompose_entropy(probs):
    eps = 1e-12
    probs = probs.clamp_min(eps)
    mean_p = probs.mean(dim=0)
    total = -(mean_p * torch.log(mean_p.clamp_min(eps))).sum(-1)
    aleatoric = -(probs * torch.log(probs)).sum(-1).mean(dim=0)
    return total, aleatoric, total - aleatoric


def calibration_error_probs(probs, y, n_bins=15):
    p = probs.mean(0)
    conf, hat = p.max(dim=-1)
    correct = (hat == y.reshape(-1)).float()
    edges = torch.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        in_bin = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if bool(in_bin.any()):
            ece += float(in_bin.float().mean()) * abs(
                float(correct[in_bin].mean()) - float(conf[in_bin].mean())
            )
    return ece

## 3.2 Fitting the members

Write `fit_ensemble`. For each member `m`:

* call `set_seed(seed + 1000 * m)` before building the model, so the
  **initialisation** differs between members;
* build a model with no dropout and fit it, passing `seed=seed + m` so the batch
  order differs too;
* keep the model in a list.

The initialisation is the part that matters. Members that start from the same
weights and differ only in the order the batches arrive end up as nearly the same
function, and the epistemic term collapses. Section 3.4 measures that.

`predict_ensemble` is then one forward pass per member, stacked into `[M, N]`.

In [ ]:
def fit_ensemble(ds, n_members=5, hidden=(64, 64), epochs=400, lr=1e-2, seed=0):
    """Train n_members networks from different initialisations. Returns a list of models."""
    # ---- TODO ------------------------------------------------------------
    # for each member m:
    #   set_seed(seed + 1000 * m)      <- different initialisation
    #   build a model with dropout=0.0 and fit it with seed=seed + m
    #   collect it
    raise NotImplementedError
    # ----------------------------------------------------------------------


@torch.no_grad()
def predict_ensemble(models, x):
    """One forward pass per member. Returns mu, sigma of shape [M, N]."""
    # ---- TODO ------------------------------------------------------------
    # each model in eval mode, one gaussian_head(model(x)) each, stack the two
    # lists into [M, N]
    raise NotImplementedError
    # ----------------------------------------------------------------------

In [ ]:
M = 5

t0 = time.perf_counter()
members = fit_ensemble(ds, n_members=M, epochs=400, lr=1e-2, seed=0)
print(f"trained {M} members in {time.perf_counter() - t0:.1f}s")
print(f"parameters per member: {count_parameters(members[0])}, total {M * count_parameters(members[0])}")

In [ ]:
# ---- check your work -------------------------------------------------------
mu_t, sd_t = predict_ensemble(members, ds.x_test)
assert len(members) == M
assert mu_t.shape == sd_t.shape == (M, len(ds.x_test)), (mu_t.shape, sd_t.shape)
assert torch.all(sd_t > 0)

# The members must be different networks, not five copies.
first = torch.cat([p.flatten() for p in members[0].parameters()])
for m in range(1, M):
    other = torch.cat([p.flatten() for p in members[m].parameters()])
    assert not torch.allclose(first, other), f"member {m} has the same weights as member 0"

_, _, epi = decompose_variance(mu_t, sd_t)
assert float(epi.mean()) > 1e-9, "the members agree everywhere: did the initialisation vary?"

print(f"OK   {M} distinct members, mean epistemic variance {float(epi.mean()):.5f}")

## 3.3 The figures

The band and the epistemic curve, as in notebook 02. With five members the outer
band is the envelope of five Gaussians rather than a smooth cloud, which is worth
noticing: the tails of a five-component mixture are described by five numbers.

In [ ]:
mu_g, sd_g = predict_ensemble(members, grid)
tot_g, ale_g, epi_g = decompose_variance(mu_g, sd_g)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
plot_band(axes[0], grid, mu_g.mean(0), ale_g.sqrt(), tot_g.sqrt(), ds=ds, title=f"ensemble, M={M}")
axes[0].legend(fontsize=8, loc="upper left")
for m in range(M):
    axes[1].plot(grid.numpy().ravel(), mu_g[m].numpy(), lw=1.4, label=f"member {m}")
axes[1].axvspan(-0.5, 1.5, color="0.9", zorder=0)
axes[1].plot(ds.x_train, ds.y_train, ".", color="0.2", ms=4, alpha=0.5)
axes[1].set_xlabel("x")
axes[1].set_ylabel("member mean")
axes[1].set_title("the members agree on the data and disagree in the gap")
axes[1].set_ylim(-4.5, 4.5)
axes[1].legend(fontsize=7)
plt.tight_layout()

In [ ]:
plot_uncertainty_vs_x({"ensemble": epi_g.sqrt()}, grid, ds);

## 3.4 Does the initialisation really matter?

Section 3.2 claimed it does. Rather than take that on trust, measure it: train a
second ensemble whose members all start from the same weights and differ only in
the order the batches arrive, then compare how much the two ensembles disagree
with themselves. Look at two places separately, because they answer different
questions: on the training data, where disagreement should be small either way,
and inside the gap, where disagreement is the whole point.

In [ ]:
def fit_ensemble_shared_init(ds, n_members=5, epochs=400, seed=0):
    """Deliberately weakened: one initialisation, different batch order."""
    models = []
    for m in range(n_members):
        set_seed(seed)  # the same initialisation every time
        model = build_model(ds, hidden=(64, 64), dropout=0.0)
        fit(
            model, ds.x_train, ds.y_train, loss_fn=default_loss(ds),
            epochs=epochs, lr=1e-2, seed=seed + m,
        )
        models.append(model)
    return models


shared = fit_ensemble_shared_init(ds, n_members=M, epochs=400)

g = grid.numpy().ravel()
in_gap = (g > -0.5) & (g < 1.5)


def epistemic_std(models, x):
    mu, sd = predict_ensemble(models, x)
    return decompose_variance(mu, sd)[2].sqrt()


for label, ms in (("shared initialisation", shared), ("varied initialisation", members)):
    on_data = float(epistemic_std(ms, ds.x_test).mean())
    gap_only = float(epistemic_std(ms, grid)[in_gap].mean())
    print(f"{label:22}  on the training data {on_data:.4f}   in the gap {gap_only:.4f}")

In the reference run the two ensembles are indistinguishable on the training data,
around 0.03 either way, and differ in the gap: about 0.053 with a shared
initialisation against 0.071 with varied ones, so roughly a third of the
disagreement in the region with no data comes from the initialisation alone.

Note what this is and is not. The disagreement does not collapse to nothing when
the initialisation is shared, because Adam driven by different batch orders still
moves the weights apart. What it loses is diversity exactly where the model is
extrapolating, which is where you were relying on it. Two hundred points of 1-D
data is also a small experiment: treat the size of the effect as a measurement on
this problem rather than a general law, and if your own numbers disagree, that is
worth a line in the report.

## 3.5 Scored

In [ ]:
mu_id, sd_id = predict_ensemble(members, ds.x_test)
mu_ood, sd_ood = predict_ensemble(members, ds.x_ood)

metrics = evaluate(
    (mu_id, sd_id),
    ds.y_test,
    (mu_ood, sd_ood),
    ds.y_ood,
    task=ds.task,
    decompose=decompose_variance,
    calibration=calibration_error,
)
metrics["train_s"] = 0.0

previous = {name: load_run("toy1d", name)["metrics"] for name in ("deterministic", "mc_dropout")}
print(results_table(previous | {"ensemble": metrics}))

## 3.6 Two things to look at

* The ensemble usually has the best RMSE of the four methods. Averaging five
  independent fits removes variance, and that is a real advantage.
* Look at `coverage@95` next to that RMSE. On several tracks the ensemble's
  nominal 95% intervals contain visibly less than 95% of the test data. Best fit
  and worst honesty, in the same row.

Two contributions to that, both quantifiable. The population variance over `M`
members underestimates the true spread by a factor `1 - 1/M`, which is 0.8 at
`M = 5`, so the epistemic standard deviation is short by about 10% before
anything else goes wrong. And each member's `sigma(x)` was fitted to that member's
own training residuals, which overfitting pushes below the true noise level.

## 3.7 Your own track

In [ ]:
TRACK = "A"  # <-- keep the same track for the whole project

ds_track = load_track(TRACK)
hp = track_hparams(TRACK, "ensemble")
print(ds_track)
print("hyperparameters:", hp)

t0 = time.perf_counter()
members_track = fit_ensemble(
    ds_track, n_members=M, epochs=int(hp["epochs"]), lr=hp["lr"], seed=0
)
train_s = time.perf_counter() - t0
print(f"trained {M} members in {train_s:.1f}s")

In [ ]:
@torch.no_grad()
def predict_probs_ensemble(models, x):
    """The classification version: one softmax vector per member, [M, N, K]."""
    out = []
    for model in models:
        model.eval()
        out.append(torch.softmax(model(x), dim=-1))
    return torch.stack(out)


if ds_track.task == "regression":
    row = evaluate(
        predict_ensemble(members_track, ds_track.x_test),
        ds_track.y_test,
        predict_ensemble(members_track, ds_track.x_ood),
        ds_track.y_ood,
        task=ds_track.task,
        decompose=decompose_variance,
        calibration=calibration_error,
    )
else:
    # Uses the two functions you wrote in section 2.8.
    row = evaluate(
        (predict_probs_ensemble(members_track, ds_track.x_test),),
        ds_track.y_test,
        (predict_probs_ensemble(members_track, ds_track.x_ood),),
        None,
        task=ds_track.task,
        decompose=decompose_entropy,
        calibration=calibration_error_probs,
    )

row["train_s"] = round(train_s, 2)
prev_track = {n: load_run(TRACK, n)["metrics"] for n in ("deterministic", "mc_dropout")}
print(results_table(prev_track | {"ensemble": row}))
save_run(TRACK, "ensemble", row)

In [ ]:
levels = np.linspace(0.05, 0.95, 12)
coverage = [interval_coverage(mu_id, sd_id, ds.y_test, float(q)) for q in levels]
plot_reliability({"ensemble": (levels, coverage)});

save_run(
    "toy1d",
    "ensemble",
    metrics,
    mean=mu_g.mean(0),
    sd_aleatoric=ale_g.sqrt(),
    sd_total=tot_g.sqrt(),
    epistemic_std=epi_g.sqrt(),
    levels=levels,
    coverage=np.array(coverage),
)

## Done when

* the check cell prints `OK`, with five distinct members;
* the shared-initialisation comparison in 3.4 shows a clear difference;
* `results/toy1d/ensemble.json` and `results/<your track>/ensemble.json` exist.

Next: notebook 04, the one built from the mathematics.